Импорт библиотек, загрузка датасета, разделение выборки, пайплайн препроцессинга

In [45]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv('../data/AmesHousing.csv')

X, y = df.drop(columns=['SalePrice']), df['SalePrice']
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, random_state=42)

num_pipe = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
cat_pipe = make_pipeline(SimpleImputer(strategy='constant', fill_value='Missing'),
                         OneHotEncoder(handle_unknown='ignore'))

pipeline = ColumnTransformer(transformers=[
    ('nums', num_pipe, make_column_selector(dtype_include=np.number)),
    ('cats', cat_pipe, make_column_selector(dtype_include=object))])

cv = KFold(n_splits=5, shuffle=True, random_state=0)

Ridge-model test

In [56]:
results = []
mae_std = 0
for a in np.logspace(1, 3, 20):
    ridge = make_pipeline(pipeline, Ridge(alpha=a))
    res = cross_validate(ridge, Xtr, ytr,
                           scoring={'mae': 'neg_mean_absolute_error', 'r2': 'r2'},
                           cv=cv, return_train_score=True)
    results.append({'alpha': a, 'mae_train': -res['train_mae'].mean(),
                    'r2_train': res['train_r2'].mean(),
                    'mae_test': -res['test_mae'].mean(),
                    'r2_test': res['test_r2'].mean()})
    if a.round(6) == 20.691381:
        mae_std = res['test_mae'].std(ddof=1)
results = pd.DataFrame(results).sort_values('mae_test', ascending=True)
print(results)
# При первом прогоне при alpha in np.logspace(-2, 3, 20) модель выдала хорошие результаты,\
# лучший mae был достигнут при alpha = 26.366509
# Подберу alpha в более узком диапазоне чтобы найти лучшее

          alpha     mae_train  r2_train      mae_test   r2_test
3     20.691381  14553.147854  0.909327  16288.472583  0.851066
2     16.237767  14466.610998  0.911399  16295.727493  0.851568
4     26.366509  14647.261002  0.907183  16298.703627  0.850540
1     12.742750  14388.833717  0.913419  16315.896331  0.851904
5     33.598183  14751.192555  0.904946  16319.241571  0.849776
0     10.000000  14316.066447  0.915410  16338.899990  0.852122
6     42.813324  14870.655289  0.902597  16347.322842  0.848939
7     54.555948  15001.423812  0.900113  16394.453334  0.847930
8     69.519280  15147.784241  0.897476  16455.205540  0.846777
9     88.586679  15309.127763  0.894665  16531.787369  0.845523
10   112.883789  15481.530718  0.891654  16624.401749  0.844096
11   143.844989  15669.991166  0.888415  16740.044387  0.842510
12   183.298071  15878.598326  0.884912  16878.056507  0.840781
13   233.572147  16106.767073  0.881100  17037.438509  0.838867
14   297.635144  16357.315130  0.876929 

In [69]:
# Поиск наибольшего alpha по 1-SE rule для упрощения модели
se = mae_std/np.sqrt(5)
threshold = 16288.472583 + se
in_threshold = results[results['mae_test'] <= threshold].sort_values('alpha', ascending=False)
print(in_threshold.head())
# По 1-SE rule наибольший допустимый alpha = 183.298071, но в данном случае я не вижу смысла жертвовать\
# лучшей статистикой ради большего alpha

         alpha     mae_train  r2_train      mae_test   r2_test
12  183.298071  15878.598326  0.884912  16878.056507  0.840781
11  143.844989  15669.991166  0.888415  16740.044387  0.842510
10  112.883789  15481.530718  0.891654  16624.401749  0.844096
9    88.586679  15309.127763  0.894665  16531.787369  0.845523
8    69.519280  15147.784241  0.897476  16455.205540  0.846777


In [ ]:
# Лучшая итоговая конфигурация Ridge-модели
ridge = make_pipeline(pipeline, Ridge(alpha=20.691381)).fit(Xtr, ytr)
pred = ridge.predict(Xte)
print(f'mae = {mean_absolute_error(yte, pred)}, r2 = {r2_score(yte, pred)}')

mae = 16388.651795631256, r2 = 0.8947479040770614


Linear-model test

In [71]:
lin = make_pipeline(pipeline, LinearRegression()).fit(Xtr, ytr)
res = cross_validate(lin, Xtr, ytr, cv=cv,
                     scoring={'mae': 'neg_mean_absolute_error', 'r2': 'r2'},
                     return_train_score=True)
print(f'mae_train = {-res['train_mae'].mean().round(6)}, r2_train = {res['train_r2'].mean().round(6)}\n\
mae_test = {-res['test_mae'].mean().round(6)}, r2_test = {res['test_r2'].mean().round(6)}')

mae_train = 12784.154728, r2_train = 0.94348
mae_test = 16662.182839, r2_test = 0.832985
